# ODI to Databricks Migration: HR.TRG_DEP INSERT

**Conversion Timestamp:** 2024-07-30 12:00:00

**Description:** This notebook processes an initial load of department data into the `trg_dep` target table.

In [ ]:
dbutils.widgets.text("ETL_JOB_TYPE", "", "1. ETL Job Type (e.g., I for incremental)")
dbutils.widgets.text("DATASOURCE_NUM_ID", "", "2. Data Source Number ID")
dbutils.widgets.text("ETL_PROC_WID", "", "3. ETL Process ID")
dbutils.widgets.text("ODI_SESS_NO", "", "4. ODI Session Number")

# ETL Parameters

In [ ]:
%sql
-- SCEN_TASK_NO {10}: Get ETL last extract time - Placeholder as no specific logic found in source.
-- In a real scenario, this would query a metadata table.
CREATE OR REPLACE TEMPORARY VIEW v_etl_last_extract_time AS
SELECT to_timestamp('1900-01-01', 'yyyy-MM-dd') AS etl_last_extract_time;

In [ ]:
%sql
-- SCEN_TASK_NO {20}: Get ETL current extract time - Placeholder as no specific logic found in source.
CREATE OR REPLACE TEMPORARY VIEW v_etl_current_extract_time AS
SELECT current_timestamp() AS etl_current_extract_time;

In [ ]:
display(spark.sql("SELECT '${ETL_JOB_TYPE}' AS ETL_JOB_TYPE, ${DATASOURCE_NUM_ID} AS DATASOURCE_NUM_ID, ${ETL_PROC_WID} AS ETL_PROC_WID, '${ODI_SESS_NO}' AS ODI_SESS_NO, etl_last_extract_time, etl_current_extract_time FROM v_etl_last_extract_time, v_etl_current_extract_time"))

# Target Table Load

In [ ]:
%sql
-- SCEN_TASK_NO {30}: Insert into HR.TRG_DEP
INSERT 
  INTO workspace.hr.trg_dep
  (
    department_id ,
    department_name ,
    manager_id ,
    location_id 
  ) 
SELECT 
  DEPARTMENTS.department_id ,
  DEPARTMENTS.department_name ,
  DEPARTMENTS.manager_id ,
  DEPARTMENTS.location_id  
FROM 
  workspace.hr.departments AS DEPARTMENTS;

In [ ]:
%sql
SELECT COUNT(*) AS record_count FROM workspace.hr.trg_dep;

# Cleanup

In [ ]:
%sql
-- No temporary staging or flow tables were used in this simple insert, so no cleanup needed here.

# Validation

In [ ]:
%sql
SELECT 
  department_id,
  department_name,
  manager_id,
  location_id
FROM 
  workspace.hr.trg_dep
ORDER BY 
  department_id
LIMIT 10;

# Conversion Notes and Manual Actions Required

1.  **Target Table DDL:** The original ODI SQL did not include the `CREATE TABLE` DDL for `HR.TRG_DEP`. Ensure the `workspace.hr.trg_dep` table is created in Databricks with appropriate Spark SQL data types (e.g., `BIGINT` for IDs, `STRING` for names) and `USING DELTA` clause. Example DDL:
    ```sql
    CREATE TABLE IF NOT EXISTS workspace.hr.trg_dep (
        department_id   BIGINT,
        department_name STRING,
        manager_id      BIGINT,
        location_id     BIGINT
    ) USING DELTA;
    ```
2.  **Schema and Table Names:** Original Oracle schema `HR` and table names `TRG_DEP`, `DEPARTMENTS` have been converted to lowercase and prefixed with `workspace.` (`workspace.hr.trg_dep`, `workspace.hr.departments`).
3.  **Oracle Hints:** The `/*+ APPEND PARALLEL */` hint has been removed as it is specific to Oracle and not applicable to Databricks Delta Lake.
4.  **ETL Parameters:** Placeholder widgets have been added, though not directly used in this simple `INSERT` statement. They follow the standard structure required by the migration framework.